In [4]:
#Вариант 5
import pandas as pd
import numpy as np

order_files = [
    "orders_2024_01.csv",
    "orders_2024_02.csv",
    "orders_2024_03.csv"
]

df_orders = pd.concat(
    [pd.read_csv(file) for file in order_files],
    ignore_index=True
)

df_events_records = pd.read_json(
    "events_records.json",
    orient="columns"
).T

df_events_records.columns = [
    "event_id", "user_id", "event_time", "event_type", "page"
]

df_events_lines = pd.read_json("events_lines.json", lines=True)

df_events = pd.concat(
    [df_events_records, df_events_lines],
    ignore_index=True
)

print("Таблица заказов:")
print(df_orders.head())

print("\nТаблица событий:")
print(df_events.head())

Таблица заказов:
   order_id  user_id       order_datetime  product_id     price  quantity  \
0         1     1044  2024-01-03 18:39:26          26  14038.15         1   
1         2     1079  2024-01-03 17:45:43          16   9172.60         1   
2         3     1041  2024-01-06 22:46:38          25   9035.26         2   
3         4     1083  2024-01-08 21:03:51           9  15234.33         3   
4         5     1068  2024-01-03 23:26:53          24   4134.38         2   

     status  
0  refunded  
1      paid  
2      paid  
3      paid  
4      paid  

Таблица событий:
  event_id user_id           event_time   event_type         page
0        1    1085  2024-01-30T20:37:32    page_view  /product/26
1        2    1006  2024-02-24T22:55:13  add_to_cart  /product/28
2        3    1024  2024-02-14T17:49:43        click  /product/27
3        4    1062  2024-02-16T11:21:32        click   /product/1
4        5    1012  2024-02-08T05:25:21  add_to_cart            /


In [5]:
df_orders["revenue"] = df_orders["price"] * df_orders["quantity"]

user_orders = df_orders.groupby("user_id").agg(
    orders_count=("order_id", "nunique"),
    revenue_sum=("revenue", "sum")
)

user_events = df_events.groupby("user_id").agg(
    events_count=("event_id", "count")
)

user_stats = user_orders.join(user_events, how="outer").fillna(0)

print("\nЗадание 1. Агрегаты по пользователям:")
print(user_stats.head())


Задание 1. Агрегаты по пользователям:
         orders_count  revenue_sum  events_count
user_id                                         
1001              177   4435767.09            54
1002              123   2830004.52            41
1003              156   4025266.18            39
1004              147   3788030.56            37
1005              141   3185653.37            44


In [6]:
features = user_stats[["orders_count", "revenue_sum", "events_count"]]

features_minmax = (
    (features - features.min()) /
    (features.max() - features.min())
)

print("\nЗадание 2. Мин–макс нормализация признаков:")
print(features_minmax.head())


Задание 2. Мин–макс нормализация признаков:
         orders_count  revenue_sum  events_count
user_id                                         
1001         1.000000     0.828116         0.750
1002         0.018182     0.000000         0.425
1003         0.618182     0.616415         0.375
1004         0.454545     0.494069         0.325
1005         0.345455     0.183414         0.500


In [7]:
max_users = features_minmax.idxmax()

print("\nЗадание 3. Пользователи с максимальными нормализованными значениями:")

for feature in features_minmax.columns:
    user_id = max_users[feature]
    value = features_minmax.loc[user_id, feature]
    print(f"{feature}: пользователь {user_id}, значение {value:.2f}")


Задание 3. Пользователи с максимальными нормализованными значениями:
orders_count: пользователь 1001, значение 1.00
revenue_sum: пользователь 1066, значение 1.00
events_count: пользователь 1091, значение 1.00
